In [7]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
from datetime import datetime

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

# 나이 계산
def calculate_age(birthdate):
    birth = datetime.strptime(birthdate, "%Y-%m-%d")
    today = datetime.today()
    age = today.year - birth.year
    if (today.month, today.day) < (birth.month, birth.day):
        age -= 1
    return age

# 환율 계산
def convert_currency(amount, rate=1508):
    return amount * rate

# bmi 계산
def calculate_bmi(height, weight):
    height_m = height / 100
    bmi = weight / (height_m ** 2)
    return round(bmi, 2)




In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_age",
            "description": "생년월일을 기준으로 만 나이를 계산",
            "parameters": {
                "type": "object",
                "properties": {
                    "birthdate": {"type": "string"}
                },
                "required": ["birthdate"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "환율 적용하여 금액 계산",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number"},
                    "rate": {"type": "number"}
                },
                "required": ["amount", "from_currency", "to_currency", "rate"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_bmi",
            "description": "키와 몸무게로 BMI 계산",
            "parameters": {
                "type": "object",
                "properties": {
                    "height": {"type": "number"},
                    "weight": {"type": "number"}
                },
                "required": ["height", "weight"]
            }
        }
    }
]


In [10]:
#입력 예제
user_input = "함수 호출이 아니다"
print(user_input)

response = client.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": "필요한 함수를 호출"}, # 모델 행동 지침
        {"role": "user", "content": user_input} # 입력 전달
    ],
    tools=tools, # 함수 목록
    tool_choice="auto" # 모델이 자동으로 함수 호출 결정
)

message = response.choices[0].message #응답 메세지 추출 

if message.tool_calls: # 함수 호출 확인
    tool_call = message.tool_calls[0] # 첫번째 함수 정보
    function_name = tool_call.function.name # 호출함수 이름 추출
    arguments = json.loads(tool_call.function.arguments) #JSON -> Dict로 변환

    if function_name == "calculate_age": # 함수 이름에 맞는 함수 실행
        result = calculate_age(**arguments) # 딕셔너리 -> 키워드로 전달

    elif function_name == "convert_currency":
        result = convert_currency(**arguments)
    
    elif function_name == "calculate_bmi":
        result = calculate_bmi(**arguments)

    print("결과:", result) # 결과출력
else:
    print(message.content) # 함수호출이 아닐때 출력

함수 호출이 아니다
알겠습니다. 함수는 호출하지 않겠습니다.

원하시는 내용을 그냥 텍스트로 답변해드릴게요.
